In [ ]:
# Environment info: useful for Colab/local reproducibility
import sys, platform, numpy as np
def where_am_i():
    try:
        import google.colab
        return "Google Colab"
    except ImportError:
        return "Local/other Jupyter"
print(f"Environment: {where_am_i()}")
print(f"Python: {sys.version.split()[0]}")
print(f"Numpy: {np.__version__}")

# Protein sequence alignment demo (global alignment)

## Main function (what you will reuse)
- `align_query_to_many(query, targets, ...)` aligns **one amino-acid query** against **many** targets and returns the top matches.
- Uses a simple global alignment (Needleman–Wunsch) with match/mismatch/gap scoring.

## Internals (what happens under the hood)
- Dynamic programming builds a **score matrix** (best score for every prefix).
- A **traceback** through pointers reconstructs the final aligned strings.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Sequence, Literal, Any

import numpy as np

In [2]:
@dataclass(frozen=True)
class AlignmentResult:
    target_id: str
    score: int
    aligned_query: str
    aligned_target: str


def _validate_protein(seq: str, *, name: str = "sequence") -> str:
    if not isinstance(seq, str) or not seq:
        raise ValueError(f"{name} must be a non-empty string")
    seq = seq.strip().upper()
    if not seq:
        raise ValueError(f"{name} must not be empty/whitespace")
    # Allow 20 AA + common ambiguity/stop symbols. Keep it permissive for teaching.
    allowed = set("ACDEFGHIKLMNPQRSTVWYBXZJUO*-" )
    bad = sorted({c for c in seq if c not in allowed})
    if bad:
        raise ValueError(f"{name} contains invalid characters: {bad}")
    return seq


def needleman_wunsch(
    query: str,
    target: str,
    *,
    match: int = 1,
    mismatch: int = -1,
    gap: int = -2,
    tie_break: Sequence[Literal["diag", "up", "left"]] = ("diag", "up", "left"),
    return_matrices: bool = False,
) -> tuple[AlignmentResult, dict[str, Any] | None]:
    """Global alignment (Needleman–Wunsch) with simple match/mismatch/gap scoring.

    Returns:
      - AlignmentResult(score, aligned strings)
      - optional debug dict containing score matrix + traceback pointers
    """
    q = _validate_protein(query, name="query")
    t = _validate_protein(target, name="target")
    n, m = len(q), len(t)

    score = np.zeros((n + 1, m + 1), dtype=int)
    pointer = np.empty((n + 1, m + 1), dtype=object)  # 'diag'/'up'/'left'/None

    # init
    pointer[0, 0] = None
    for i in range(1, n + 1):
        score[i, 0] = score[i - 1, 0] + gap
        pointer[i, 0] = "up"
    for j in range(1, m + 1):
        score[0, j] = score[0, j - 1] + gap
        pointer[0, j] = "left"

    # fill
    def s(a: str, b: str) -> int:
        return match if a == b else mismatch

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diag = score[i - 1, j - 1] + s(q[i - 1], t[j - 1])
            up = score[i - 1, j] + gap
            left = score[i, j - 1] + gap
            candidates = {"diag": diag, "up": up, "left": left}
            best = max(candidates.values())
            # deterministic tie break for reproducibility
            for move in tie_break:
                if candidates[move] == best:
                    pointer[i, j] = move
                    break
            score[i, j] = best

    # traceback
    i, j = n, m
    aq: list[str] = []
    at: list[str] = []
    while i > 0 or j > 0:
        move = pointer[i, j]
        if move == "diag":
            aq.append(q[i - 1])
            at.append(t[j - 1])
            i -= 1
            j -= 1
        elif move == "up":
            aq.append(q[i - 1])
            at.append("-")
            i -= 1
        elif move == "left":
            aq.append("-")
            at.append(t[j - 1])
            j -= 1
        else:
            # Should only happen at (0,0)
            break

    aq_str = "".join(reversed(aq))
    at_str = "".join(reversed(at))

    result = AlignmentResult(
        target_id="",
        score=int(score[n, m]),
        aligned_query=aq_str,
        aligned_target=at_str,
    )
    debug = None
    if return_matrices:
        debug = {
            "query": q,
            "target": t,
            "score": score,
            "pointer": pointer,
            "params": {"match": match, "mismatch": mismatch, "gap": gap, "tie_break": list(tie_break)},
        }
    return result, debug

In [3]:
def align_query_to_many(
    query: str,
    targets: Sequence[str] | dict[str, str],
    *,
    match: int = 1,
    mismatch: int = -1,
    gap: int = -2,
    top_k: int = 5,
    return_alignments: bool = True,
    tie_break: Sequence[Literal["diag", "up", "left"]] = ("diag", "up", "left"),
) -> list[AlignmentResult]:
    """Align one query to many targets and return the best matches.

    `targets` can be:
      - list/tuple of sequences (ids will be 't0', 't1', ...)
      - dict of {id: sequence}
    """
    if top_k <= 0:
        raise ValueError("top_k must be >= 1")

    q = _validate_protein(query, name="query")
    if isinstance(targets, dict):
        items = list(targets.items())
    else:
        items = [(f"t{i}", s) for i, s in enumerate(targets)]

    results: list[AlignmentResult] = []
    for tid, tseq in items:
        res, _ = needleman_wunsch(
            q,
            tseq,
            match=match,
            mismatch=mismatch,
            gap=gap,
            tie_break=tie_break,
            return_matrices=False,
        )
        if not return_alignments:
            res = AlignmentResult(target_id=tid, score=res.score, aligned_query="", aligned_target="")
        else:
            res = AlignmentResult(target_id=tid, score=res.score, aligned_query=res.aligned_query, aligned_target=res.aligned_target)
        results.append(res)

    results.sort(key=lambda r: r.score, reverse=True)
    return results[: min(top_k, len(results))]

In [4]:
# Quick usage: align one query against many targets
query = "PAWHEAE"
targets = {
    "seq1": "HEAGAWGHEE",
    "seq2": "PAWHEAE",
    "seq3": "PAWHEA--",  # includes gaps '-' just to show validator allows it
}

hits = align_query_to_many(query, targets, match=2, mismatch=-1, gap=-2, top_k=3)
for h in hits:
    print(h.target_id, "score=", h.score)
    print(h.aligned_query)
    print(h.aligned_target)
    print()

seq2 score= 14
PAWHEAE
PAWHEAE

seq3 score= 9
PAWHEA-E
PAWHEA--

seq1 score= -1
---PAW-HEAE
HEAGAWGHE-E



In [5]:
# Show how it works internally on ONE pair (score matrix + traceback)
q = "PAWHEAE"
t = "HEAGAWGHEE"

res, dbg = needleman_wunsch(q, t, match=2, mismatch=-1, gap=-2, return_matrices=True)
print("Final score:", res.score)
print(res.aligned_query)
print(res.aligned_target)

score = dbg["score"]
pointer = dbg["pointer"]

print("\nScore matrix shape:", score.shape)
print(score)

Final score: -1
---PAW-HEAE
HEAGAWGHE-E

Score matrix shape: (8, 11)
[[  0  -2  -4  -6  -8 -10 -12 -14 -16 -18 -20]
 [ -2  -1  -3  -5  -7  -9 -11 -13 -15 -17 -19]
 [ -4  -3  -2  -1  -3  -5  -7  -9 -11 -13 -15]
 [ -6  -5  -4  -3  -2  -4  -3  -5  -7  -9 -11]
 [ -8  -4  -6  -5  -4  -3  -5  -4  -3  -5  -7]
 [-10  -6  -2  -4  -6  -5  -4  -6  -5  -1  -3]
 [-12  -8  -4   0  -2  -4  -6  -5  -7  -3  -2]
 [-14 -10  -6  -2  -1  -3  -5  -7  -6  -5  -1]]


In [6]:
# Traceback path visualization (coordinates + moves)
def traceback_path(pointer: np.ndarray) -> list[tuple[int, int, str | None]]:
    i, j = pointer.shape[0] - 1, pointer.shape[1] - 1
    path: list[tuple[int, int, str | None]] = [(i, j, pointer[i, j])]
    while i > 0 or j > 0:
        move = pointer[i, j]
        if move == "diag":
            i -= 1; j -= 1
        elif move == "up":
            i -= 1
        elif move == "left":
            j -= 1
        else:
            break
        path.append((i, j, pointer[i, j]))
    return list(reversed(path))

path = traceback_path(pointer)
print("Path length:", len(path))
print("First 10 steps (i,j,move):")
for step in path[:10]:
    print(step)

Path length: 12
First 10 steps (i,j,move):
(0, 0, None)
(0, 1, 'left')
(0, 2, 'left')
(0, 3, 'left')
(1, 4, 'diag')
(2, 5, 'diag')
(3, 6, 'diag')
(3, 7, 'left')
(4, 8, 'diag')
(5, 9, 'diag')


In [7]:
# Pretty-print the score matrix with row/col labels (small examples)
def format_score_matrix(score: np.ndarray, query: str, target: str) -> str:
    q = "-" + query
    t = "-" + target
    cell_w = max(3, max(len(str(int(x))) for x in score.flatten()) + 1)
    header = "".ljust(cell_w) + "".join(ch.rjust(cell_w) for ch in t)
    lines = [header]
    for i, ch in enumerate(q):
        row = ch.rjust(cell_w) + "".join(str(int(score[i, j])).rjust(cell_w) for j in range(len(t)))
        lines.append(row)
    return "\n".join(lines)

print(format_score_matrix(score, dbg["query"], dbg["target"]))

       -   H   E   A   G   A   W   G   H   E   E
   -   0  -2  -4  -6  -8 -10 -12 -14 -16 -18 -20
   P  -2  -1  -3  -5  -7  -9 -11 -13 -15 -17 -19
   A  -4  -3  -2  -1  -3  -5  -7  -9 -11 -13 -15
   W  -6  -5  -4  -3  -2  -4  -3  -5  -7  -9 -11
   H  -8  -4  -6  -5  -4  -3  -5  -4  -3  -5  -7
   E -10  -6  -2  -4  -6  -5  -4  -6  -5  -1  -3
   A -12  -8  -4   0  -2  -4  -6  -5  -7  -3  -2
   E -14 -10  -6  -2  -1  -3  -5  -7  -6  -5  -1
